# 01 — Creación del Dataset Maestro Granada

Objetivo: construir una primera versión del dataset maestro del proyecto **CoopScore Granada** a partir de datos GIS de suelo residencial y datos oficiales de población municipal.

Entradas esperadas:

- `data/raw/suelo_granada.csv`
- `data/raw/suelo_granada_clasificado.csv`
- `data/raw/suelo_granada_residencial.csv`
- `data/raw/2871.xlsx`

Salidas generadas:

- `data/processed/municipios_granada.csv`
- `data/processed/suelo_residencial_municipio.csv`
- `data/processed/dataset_maestro_granada_v1.csv`


In [151]:
# ============================================================
# 0. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Rutas del proyecto
BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print('Directorio base:', BASE_DIR)
print('Raw existe:', DATA_RAW.exists())
print('Processed existe:', DATA_PROCESSED.exists())

Directorio base: c:\CoopScore_Granada
Raw existe: True
Processed existe: True


In [152]:
# ============================================================
# 1. CARGA DE DATOS RAW
# ============================================================

suelo = pd.read_csv(DATA_RAW / 'suelo_granada.csv')
suelo_clasificado = pd.read_csv(DATA_RAW / 'suelo_granada_clasificado.csv')
suelo_residencial = pd.read_csv(DATA_RAW / 'suelo_granada_residencial.csv')
ine_raw = pd.read_csv(
    DATA_RAW / '2871.csv',
    sep='\t',
    header=None
)

print('suelo:', suelo.shape)
print('suelo_clasificado:', suelo_clasificado.shape)
print('suelo_residencial:', suelo_residencial.shape)
print('ine_raw:', ine_raw.shape)

suelo: (262, 5)
suelo_clasificado: (2000, 7)
suelo_residencial: (345, 7)
ine_raw: (176, 1)


In [153]:
# Vista rápida de los datos GIS residenciales
suelo_residencial.head()

,OBJECTID,codigo_ine,UGL_SG,NOMBRE,area_m2,SHAPE.STLength(),uso_suelo
0,1,18001,2.00,NaN,"62,405.08","1,185.38",Residencial
1,2,18002,2.00,NaN,"150,962.61","2,758.68",Residencial
2,5,18004,2.00,NaN,"213,613.19","1,948.35",Residencial
3,10,18006,2.00,NaN,"7,759.43",371.90,Residencial
4,13,18006,2.00,NaN,"10,586.52",492.23,Residencial


In [154]:
# Vista rápida del Excel INE
ine_raw.head(20)

,0
0,Municipios;Sexo;Periodo;Total
1,18001 Agrón;Total;2025;247
2,18002 Alamedilla;Total;2025;518
3,18003 Albolote;Total;2025;19.768
4,18004 Albondón;Total;2025;703
5,18006 Albuñol;Total;2025;7.388
6,18007 Albuñuelas;Total;2025;794
7,18005 Albuñán;Total;2025;422
8,18010 Aldeire;Total;2025;597
9,18011 Alfacar;Total;2025;5.834


In [155]:
print("Shape:", ine_raw.shape)
print("\nColumnas:")
print(ine_raw.columns)

print("\nPrimeras 15 filas:")
print(ine_raw.head(15))

Shape: (176, 1)

Columnas:
Index([0], dtype='int64')

Primeras 15 filas:
                                           0
0              Municipios;Sexo;Periodo;Total
1                 18001 Agrón;Total;2025;247
2            18002 Alamedilla;Total;2025;518
3           18003 Albolote;Total;2025;19.768
4              18004 Albondón;Total;2025;703
5             18006 Albuñol;Total;2025;7.388
6            18007 Albuñuelas;Total;2025;794
7               18005 Albuñán;Total;2025;422
8               18010 Aldeire;Total;2025;597
9             18011 Alfacar;Total;2025;5.834
10         18012 Algarinejo;Total;2025;2.335
11  18013 Alhama de Granada;Total;2025;5.544
12          18014 Alhendín;Total;2025;10.475
13     18015 Alicún de Ortega;Total;2025;461
14            18016 Almegíjar;Total;2025;327


In [156]:
# Comprobación básica de municipios
print('Municipios INE cargados:', municipios.shape[0])
print('Duplicados codigo_ine:', municipios['codigo_ine'].duplicated().sum())
municipios.sample(10, random_state=42)

Municipios INE cargados: 174
Duplicados codigo_ine: 0


,codigo_ine,municipio,poblacion_2025
157,18914,Valderrubio,2069
146,18174,Santa Cruz del Comercio,523
103,18121,Lobras,139
129,18152,Pedro Martínez,1166
142,18168,Quéntar,952
140,18159,Píñar,1040
43,18048,Cijuela,3827
16,18904,Alpujarra de la Sierra,933
128,18151,Pampaneira,312
66,18072,Escúzar,826


In [157]:
# ============================================================
# 2. LIMPIEZA DATOS INE
# ============================================================

# Separar la única columna usando ";"
ine = ine_raw[0].str.split(";", expand=True)

ine.columns = [
    "municipio_raw",
    "sexo",
    "periodo",
    "poblacion_2025"
]

# Eliminar cabecera
ine = ine.iloc[1:].copy()

# Extraer código INE
ine["codigo_ine"] = ine["municipio_raw"].str.extract(r"^(\d{5})")

# Extraer nombre municipio
ine["municipio"] = (
    ine["municipio_raw"]
    .str.replace(r"^\d{5}\s+", "", regex=True)
    .str.strip()
)

# Convertir población
ine["poblacion_2025"] = (
    ine["poblacion_2025"]
    .str.replace(".", "", regex=False)
    .astype(int)
)

# Dataset final municipios
municipios = ine[
    ["codigo_ine", "municipio", "poblacion_2025"]
].copy()

# Eliminar filas que no sean municipios reales
municipios = municipios[
    municipios["codigo_ine"].notna()
].copy()

print(municipios.head())
print(municipios.shape)

  codigo_ine   municipio  poblacion_2025
1      18001       Agrón             247
2      18002  Alamedilla             518
3      18003    Albolote           19768
4      18004    Albondón             703
5      18006     Albuñol            7388
(174, 3)


In [158]:
print(len(municipios))
print(len(suelo_resumen))
print(len(dataset))

174
128
174


In [159]:
print(suelo_resumen.columns.tolist())

['codigo_ine', 'n_poligonos_residenciales', 'superficie_residencial_m2', 'superficie_media_poligono_m2', 'superficie_mediana_poligono_m2', 'superficie_max_poligono_m2', 'perimetro_total_m', 'superficie_residencial_ha']


In [160]:
# ============================================================
# 3. LIMPIEZA Y RESUMEN DEL SUELO RESIDENCIAL
# ============================================================

res = suelo_residencial.copy()

# Normalizamos código INE a string de 5 dígitos
res['codigo_ine'] = res['codigo_ine'].astype(int).astype(str).str.zfill(5)

# Renombramos columnas técnicas
res = res.rename(columns={
    'SHAPE.STLength()': 'perimetro_m',
    'area_m2': 'superficie_poligono_m2'
})

# Aseguramos tipos numéricos
res['superficie_poligono_m2'] = pd.to_numeric(res['superficie_poligono_m2'], errors='coerce')
res['perimetro_m'] = pd.to_numeric(res['perimetro_m'], errors='coerce')

# Validaciones rápidas
print('Registros residenciales:', len(res))
print('Municipios con suelo residencial:', res['codigo_ine'].nunique())
print('Área total residencial m²:', round(res['superficie_poligono_m2'].sum(), 2))

res.head()

Registros residenciales: 345
Municipios con suelo residencial: 128
Área total residencial m²: 84520529.52


,OBJECTID,codigo_ine,UGL_SG,NOMBRE,superficie_poligono_m2,perimetro_m,uso_suelo
0,1,18001,2.00,NaN,"62,405.08","1,185.38",Residencial
1,2,18002,2.00,NaN,"150,962.61","2,758.68",Residencial
2,5,18004,2.00,NaN,"213,613.19","1,948.35",Residencial
3,10,18006,2.00,NaN,"7,759.43",371.90,Residencial
4,13,18006,2.00,NaN,"10,586.52",492.23,Residencial


In [161]:
# Agrupamos por municipio
suelo_resumen = (
    res
    .groupby('codigo_ine')
    .agg(
        n_poligonos_residenciales=('OBJECTID', 'count'),
        superficie_residencial_m2=('superficie_poligono_m2', 'sum'),
        superficie_media_poligono_m2=('superficie_poligono_m2', 'mean'),
        superficie_mediana_poligono_m2=('superficie_poligono_m2', 'median'),
        superficie_max_poligono_m2=('superficie_poligono_m2', 'max'),
        perimetro_total_m=('perimetro_m', 'sum')
    )
    .reset_index()
)

suelo_resumen['superficie_residencial_ha'] = suelo_resumen['superficie_residencial_m2'] / 10_000

suelo_resumen = suelo_resumen.sort_values('superficie_residencial_m2', ascending=False)
suelo_resumen.head(15)

,codigo_ine,n_poligonos_residenciales,superficie_residencial_m2,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,superficie_residencial_ha
120,18905,1,"3,682,034.80","3,682,034.80","3,682,034.80","3,682,034.80","40,983.79",368.20
60,18089,7,"3,460,927.70","494,418.24","118,598.31","2,848,795.58","43,493.54",346.09
16,18022,1,"3,364,039.55","3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",336.40
21,18029,9,"3,295,178.29","366,130.92","108,955.23","1,376,322.69","32,558.92",329.52
2,18003,1,"2,949,835.53","2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",294.98
78,18122,19,"2,865,887.93","150,836.21","59,364.89","1,489,210.99","54,607.58",286.59
106,18173,1,"2,549,806.21","2,549,806.21","2,549,806.21","2,549,806.21","37,748.82",254.98
66,18102,9,"2,251,831.88","250,203.54","155,067.85","826,560.63","41,696.83",225.18
17,18023,7,"2,195,843.08","313,691.87","40,544.71","1,948,584.38","42,292.24",219.58
15,18021,3,"2,018,746.32","672,915.44","122,760.38","1,777,167.56","31,524.40",201.87


In [162]:
municipios = municipios.drop_duplicates(
    subset='codigo_ine',
    keep='first'
)

In [163]:
# ============================================================
# 4. CRUCE SUELO RESIDENCIAL + MUNICIPIOS INE
# ============================================================

dataset = municipios.merge(
    suelo_resumen,
    on='codigo_ine',
    how='left'
)

# Reordenamos columnas
cols = [
    'codigo_ine',
    'municipio',
    'poblacion_2025',
    'n_poligonos_residenciales',
    'superficie_residencial_m2',
    'superficie_residencial_ha',
    'superficie_media_poligono_m2',
    'superficie_mediana_poligono_m2',
    'superficie_max_poligono_m2',
    'perimetro_total_m'
]

dataset = dataset[cols]

# ============================================================
# VARIABLES DERIVADAS
# ============================================================

dataset['suelo_residencial_m2_por_hab'] = np.where(
    dataset['poblacion_2025'] > 0,
    dataset['superficie_residencial_m2'] / dataset['poblacion_2025'],
    np.nan
)

# ============================================================
# CONTROL DE CALIDAD
# ============================================================

print(f"Municipios INE: {len(municipios)}")
print(f"Municipios suelo: {len(suelo_resumen)}")
print(f"Municipios dataset: {len(dataset)}")

print("\nGranada:")
print(
    dataset[
        dataset["municipio"] == "Granada"
    ][["municipio", "poblacion_2025"]]
)

# Orden final
dataset = dataset.sort_values(
    'superficie_residencial_m2',
    ascending=False
).reset_index(drop=True)

dataset.head()

Municipios INE: 174
Municipios suelo: 128
Municipios dataset: 174

Granada:
   municipio  poblacion_2025
76   Granada          233975


,codigo_ine,municipio,poblacion_2025,n_poligonos_residenciales,superficie_residencial_m2,superficie_residencial_ha,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,suelo_residencial_m2_por_hab
0,18905,"Gabias, Las",23584,1.00,"3,682,034.80",368.20,"3,682,034.80","3,682,034.80","3,682,034.80","40,983.79",156.12
1,18089,Guadix,18881,7.00,"3,460,927.70",346.09,"494,418.24","118,598.31","2,848,795.58","43,493.54",183.30
2,18022,Atarfe,20914,1.00,"3,364,039.55",336.40,"3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",160.85
3,18029,Benamaurel,2235,9.00,"3,295,178.29",329.52,"366,130.92","108,955.23","1,376,322.69","32,558.92","1,474.35"
4,18003,Albolote,19768,1.00,"2,949,835.53",294.98,"2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",149.22


In [164]:
# ============================================================
# 5. CONTROL DE CALIDAD DEL CRUCE
# ============================================================

sin_nombre = dataset[dataset['municipio'].isna()]
print('Municipios sin nombre tras cruce:', len(sin_nombre))

if len(sin_nombre) > 0:
    display(sin_nombre[['codigo_ine', 'superficie_residencial_m2', 'n_poligonos_residenciales']])
else:
    print('Cruce correcto: todos los códigos INE del suelo residencial tienen municipio asociado.')

Municipios sin nombre tras cruce: 0
Cruce correcto: todos los códigos INE del suelo residencial tienen municipio asociado.


In [165]:
# Municipios clave para la demo
municipios_demo = [
    'Granada', 'Motril', 'Salobreña', 'Almuñécar',
    'Armilla', 'Maracena', 'Albolote', 'Atarfe',
    'Las Gabias', 'Ogíjares', 'Loja', 'Guadix', 'Baza'
]

# Búsqueda flexible por si hay tildes o nombres compuestos diferentes
patron = '|'.join(municipios_demo)

demo = dataset[dataset['municipio'].fillna('').str.contains(patron, case=False, regex=True)].copy()
demo.sort_values('superficie_residencial_m2', ascending=False)

,codigo_ine,municipio,poblacion_2025,n_poligonos_residenciales,superficie_residencial_m2,superficie_residencial_ha,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,suelo_residencial_m2_por_hab
1,18089,Guadix,18881,7.00,"3,460,927.70",346.09,"494,418.24","118,598.31","2,848,795.58","43,493.54",183.30
2,18022,Atarfe,20914,1.00,"3,364,039.55",336.40,"3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",160.85
4,18003,Albolote,19768,1.00,"2,949,835.53",294.98,"2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",149.22
5,18122,Loja,20951,19.00,"2,865,887.93",286.59,"150,836.21","59,364.89","1,489,210.99","54,607.58",136.79
6,18173,Salobreña,12760,1.00,"2,549,806.21",254.98,"2,549,806.21","2,549,806.21","2,549,806.21","37,748.82",199.83
8,18023,Baza,20587,7.00,"2,195,843.08",219.58,"313,691.87","40,544.71","1,948,584.38","42,292.24",106.66
9,18021,Armilla,25300,3.00,"2,018,746.32",201.87,"672,915.44","122,760.38","1,777,167.56","31,524.40",79.79
12,18017,Almuñécar,27544,14.00,"1,697,002.90",169.70,"121,214.49","70,834.09","434,641.19","39,058.33",61.61
19,18127,Maracena,22294,4.00,"1,233,802.70",123.38,"308,450.67","31,368.64","1,169,108.11","23,592.45",55.34
25,18053,Cortes de Baza,1769,8.00,"959,226.33",95.92,"119,903.29","100,042.67","383,662.37","25,198.71",542.24


In [166]:
# ============================================================
# 6. EXPORTACIÓN DE RESULTADOS
# ============================================================

municipios.to_csv(DATA_PROCESSED / 'municipios_granada.csv', index=False, encoding='utf-8-sig')
suelo_resumen.to_csv(DATA_PROCESSED / 'suelo_residencial_municipio.csv', index=False, encoding='utf-8-sig')
dataset.to_csv(DATA_PROCESSED / 'dataset_maestro_granada_v1.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('-', DATA_PROCESSED / 'municipios_granada.csv')
print('-', DATA_PROCESSED / 'suelo_residencial_municipio.csv')
print('-', DATA_PROCESSED / 'dataset_maestro_granada_v1.csv')

Archivos generados:
- c:\CoopScore_Granada\data\processed\municipios_granada.csv
- c:\CoopScore_Granada\data\processed\suelo_residencial_municipio.csv
- c:\CoopScore_Granada\data\processed\dataset_maestro_granada_v1.csv


In [167]:
# ============================================================
# 7. RESUMEN EJECUTIVO PARA DOCUMENTACIÓN
# ============================================================

resumen = {
    'municipios_ine_total': municipios.shape[0],
    'poligonos_residenciales_total': res.shape[0],
    'municipios_con_suelo_residencial': dataset.shape[0],
    'superficie_residencial_total_m2': round(dataset['superficie_residencial_m2'].sum(), 2),
    'superficie_residencial_total_ha': round(dataset['superficie_residencial_ha'].sum(), 2),
    'poblacion_total_municipios_con_suelo': int(dataset['poblacion_2025'].sum())
}

resumen

{'municipios_ine_total': 174,
 'poligonos_residenciales_total': 345,
 'municipios_con_suelo_residencial': 174,
 'superficie_residencial_total_m2': np.float64(84520529.52),
 'superficie_residencial_total_ha': np.float64(8452.05),
 'poblacion_total_municipios_con_suelo': 942618}

In [168]:
df = pd.read_csv("../data/processed/dataset_maestro_granada_v1.csv")

df[["municipio","poblacion_2025"]].head()

,municipio,poblacion_2025
0,"Gabias, Las",23584
1,Guadix,18881
2,Atarfe,20914
3,Benamaurel,2235
4,Albolote,19768


In [169]:
dataset["poblacion_2025"].describe()

count       174.00
mean      5,417.34
std      18,795.39
min         139.00
25%         618.25
50%       1,199.00
75%       3,965.75
max     233,975.00
Name: poblacion_2025, dtype: float64

In [170]:
municipios[
    municipios["municipio"].str.contains(
        "Granada",
        case=False,
        na=False
    )
]

,codigo_ine,municipio,poblacion_2025
11,18013,Alhama de Granada,5544
22,18024,Beas de Granada,1007
61,18915,Domingo Pérez de Granada,822
77,18087,Granada,233975


In [171]:
dataset[
    dataset["municipio"] == "Granada"
][["municipio","poblacion_2025"]]

,municipio,poblacion_2025
141,Granada,233975


In [172]:
dataset.sort_values(
    "poblacion_2025",
    ascending=False
)[["municipio","poblacion_2025"]].head(15)

,municipio,poblacion_2025
141,Granada,233975
37,Motril,59862
12,Almuñécar,27544
9,Armilla,25300
0,"Gabias, Las",23584
19,Maracena,22294
5,Loja,20951
2,Atarfe,20914
8,Baza,20587
173,"Zubia, La",20389


In [173]:
dataset.to_csv(
    DATA_PROCESSED / "dataset_maestro_granada_v1.csv",
    index=False
)

In [183]:
from pathlib import Path

DATA_RAW = BASE_DIR / "data" / "raw"

for f in DATA_RAW.iterdir():
    print(f.name)

2871.csv
distancia_pueblos.csv
suelo_granada.csv
suelo_granada_clasificado.csv
suelo_granada_residencial.csv
suelo_residencial_granada_SUNC_Urbanizable.csv


In [185]:
suelo_nuevo = pd.read_csv(
    DATA_RAW / "suelo_residencial_granada_SUNC_Urbanizable.csv"
)

print(suelo_nuevo.shape)
print(suelo_nuevo.columns.tolist())

suelo_nuevo.head()

(175, 4)
['codigo_ine', 'nombre', 'area_m2_SUNC', 'area_m2_Urbanizable']


,codigo_ine,nombre,area_m2_SUNC,area_m2_Urbanizable
0,18,Granada,0.00,0.00
1,18001,AgrÃ³n,0.00,0.00
2,18002,Alamedilla,0.00,0.00
3,18003,Albolote,"828,396.00","2,047,817.14"
4,18004,AlbondÃ³n,0.00,0.00


In [186]:
print(suelo_nuevo["codigo_ine"].nunique())
print(municipios["codigo_ine"].nunique())

175
174


In [187]:
codigos_comunes = set(
    suelo_nuevo["codigo_ine"]
).intersection(
    set(municipios["codigo_ine"].astype(int))
)

print(len(codigos_comunes))

0


In [188]:
print(suelo_nuevo["codigo_ine"].dtype)
print(municipios["codigo_ine"].dtype)

print(suelo_nuevo["codigo_ine"].head(10).tolist())
print(municipios["codigo_ine"].head(10).tolist())

object
object
['18', '18001', '18002', '18003', '18004', '18005', '18006', '18007', '18010', '18011']
['18001', '18002', '18003', '18004', '18006', '18007', '18005', '18010', '18011', '18012']


In [189]:
import unicodedata

def normalizar(texto):

    texto = str(texto).upper().strip()

    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    return texto

In [190]:
dataset["municipio_key"] = dataset["municipio"].apply(normalizar)

suelo_nuevo["municipio_key"] = suelo_nuevo["nombre"].apply(normalizar)

In [191]:
suelo_nuevo = suelo_nuevo[
    [
        "municipio_key",
        "area_m2_SUNC",
        "area_m2_Urbanizable"
    ]
]

In [192]:
dataset_v2 = dataset.merge(
    suelo_nuevo,
    on="municipio_key",
    how="left"
)

In [193]:
dataset_v2["area_m2_desarrollo"] = (
    dataset_v2["area_m2_SUNC"].fillna(0)
    +
    dataset_v2["area_m2_Urbanizable"].fillna(0)
)

In [194]:
dataset_v2["sunc_m2_por_hab"] = (
    dataset_v2["area_m2_SUNC"]
    /
    dataset_v2["poblacion_2025"]
)

dataset_v2["urbanizable_m2_por_hab"] = (
    dataset_v2["area_m2_Urbanizable"]
    /
    dataset_v2["poblacion_2025"]
)

In [195]:
print(dataset_v2.shape)

dataset_v2.info()

dataset_v2.head()

(175, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   codigo_ine                      175 non-null    object 
 1   municipio                       175 non-null    object 
 2   poblacion_2025                  175 non-null    int64  
 3   n_poligonos_residenciales       128 non-null    float64
 4   superficie_residencial_m2       128 non-null    float64
 5   superficie_residencial_ha       128 non-null    float64
 6   superficie_media_poligono_m2    128 non-null    float64
 7   superficie_mediana_poligono_m2  128 non-null    float64
 8   superficie_max_poligono_m2      128 non-null    float64
 9   perimetro_total_m               128 non-null    float64
 10  suelo_residencial_m2_por_hab    128 non-null    float64
 11  municipio_key                   175 non-null    object 
 12  area_m2_SUNC              

,codigo_ine,municipio,poblacion_2025,n_poligonos_residenciales,superficie_residencial_m2,superficie_residencial_ha,superficie_media_poligono_m2,superficie_mediana_poligono_m2,superficie_max_poligono_m2,perimetro_total_m,suelo_residencial_m2_por_hab,municipio_key,area_m2_SUNC,area_m2_Urbanizable,area_m2_desarrollo,sunc_m2_por_hab,urbanizable_m2_por_hab
0,18905,"Gabias, Las",23584,1.00,"3,682,034.80",368.20,"3,682,034.80","3,682,034.80","3,682,034.80","40,983.79",156.12,"GABIAS, LAS",NaN,NaN,0.00,NaN,NaN
1,18089,Guadix,18881,7.00,"3,460,927.70",346.09,"494,418.24","118,598.31","2,848,795.58","43,493.54",183.30,GUADIX,"438,023.00","353,243.00","791,266.00",23.20,18.71
2,18022,Atarfe,20914,1.00,"3,364,039.55",336.40,"3,364,039.55","3,364,039.55","3,364,039.55","37,294.92",160.85,ATARFE,"33,512.00","3,636,106.00","3,669,618.00",1.60,173.86
3,18029,Benamaurel,2235,9.00,"3,295,178.29",329.52,"366,130.92","108,955.23","1,376,322.69","32,558.92","1,474.35",BENAMAUREL,"30,085.00","446,403.00","476,488.00",13.46,199.73
4,18003,Albolote,19768,1.00,"2,949,835.53",294.98,"2,949,835.53","2,949,835.53","2,949,835.53","42,092.59",149.22,ALBOLOTE,"828,396.00","2,047,817.14","2,876,213.14",41.91,103.59


In [196]:
dataset_v2[
    [
        "municipio",
        "area_m2_SUNC",
        "area_m2_Urbanizable",
        "area_m2_desarrollo"
    ]
].sample(10)

,municipio,area_m2_SUNC,area_m2_Urbanizable,area_m2_desarrollo
170,Villamena,0.00,0.00,0.00
41,Orce,"122,920.00",0.00,"122,920.00"
2,Atarfe,"33,512.00","3,636,106.00","3,669,618.00"
21,Alfacar,"287,997.00","313,261.00","601,258.00"
84,Montillana,"181,300.00",0.00,"181,300.00"
66,Algarinejo,"81,825.00",0.00,"81,825.00"
135,Cogollos de la Vega,"69,634.00","133,693.00","203,327.00"
160,Píñar,NaN,NaN,0.00
31,Benalúa,NaN,NaN,0.00
14,Padul,"784,786.00","144,000.00","928,786.00"


In [197]:
dataset_v2[
    dataset_v2["municipio"] == "Granada"
][[
    "municipio",
    "area_m2_SUNC",
    "area_m2_Urbanizable",
    "area_m2_desarrollo"
]]

,municipio,area_m2_SUNC,area_m2_Urbanizable,area_m2_desarrollo
141,Granada,0.00,0.00,0.00
142,Granada,"1,037,518.00","3,362,687.00","4,400,205.00"


In [198]:
dataset_v2.to_csv(
    DATA_PROCESSED / "dataset_maestro_granada_v2.csv",
    index=False
)

In [199]:
import pandas as pd

check = pd.read_csv(
    DATA_PROCESSED / "dataset_maestro_granada_v2.csv"
)

print(check.shape)

(175, 17)
